In [ ]:
from kafka import KafkaConsumer
import json
from collections import defaultdict
from datetime import datetime
import mysql.connector
import csv
import os

# ---------------------------------------------------
# MYSQL CONNECTION
# ---------------------------------------------------

db = mysql.connector.connect(
    host="localhost",
    user="root",
    password="sql09",
    database="ecommerce_anomaly_db"
)

cursor = db.cursor()

print("MySQL Connected...")

# ---------------------------------------------------
# CREATE KAFKA CONSUMER
# ---------------------------------------------------

consumer = KafkaConsumer(
    'user_events',
    bootstrap_servers='localhost:9092',

    # convert kafka byte data -> python dictionary
    value_deserializer=lambda x: json.loads(
        x.decode('utf-8')
    ),

    auto_offset_reset='earliest',
    enable_auto_commit=True
)

# ---------------------------------------------------
# MEMORY STRUCTURES
# ---------------------------------------------------

user_events = defaultdict(list)
user_scores = defaultdict(float)

print("Consumer started...\n")

# ---------------------------------------------------
# OUTPUT FOLDER + CSV FILE
# ---------------------------------------------------

output_path = r"C:\Users\naman\Downloads\output"

if not os.path.exists(output_path):
    os.makedirs(output_path)

file_path = os.path.join(
    output_path,
    "alerts.csv"
)

# overwrite csv every run
with open(file_path, mode='w', newline='') as file:
    writer = csv.writer(file)

    writer.writerow([
        'user_id',
        'timestamp',
        'risk_score',
        'risk_level',
        'trigger_reason'
    ])

# ---------------------------------------------------
# MYSQL INSERT QUERY
# ---------------------------------------------------

insert_query = """
INSERT INTO anomaly_alerts (
    user_id,
    event_timestamp,
    risk_score,
    risk_level,
    reason
)
VALUES (%s, %s, %s, %s, %s)
"""

# ---------------------------------------------------
# START STREAM PROCESSING
# ---------------------------------------------------

for message in consumer:

    # ---------------------------------------------------
    # READ EVENT DATA
    # ---------------------------------------------------

    data = message.value

    user_id = data['user_id']
    event_type = data['event_type']

    timestamp = datetime.fromisoformat(
        data['timestamp']
    )

    price = data['price']
    location = data['location']
    is_logged_in = data['is_logged_in']

    # ---------------------------------------------------
    # STORE USER HISTORY
    # ---------------------------------------------------

    user_events[user_id].append({
        "event_type": event_type,
        "timestamp": timestamp,
        "price": price,
        "location": location
    })

    # ---------------------------------------------------
    # INITIALIZE SCORE
    # ---------------------------------------------------

    score = 0
    reason = []

    recent = user_events[user_id][-5:]

    # ---------------------------------------------------
    # RULE 1: RAPID PURCHASE
    # ---------------------------------------------------

    if len(recent) >= 5:

        purchase_count = sum(
            1 for e in recent
            if e['event_type'] == 'purchase'
        )

        time_diff = (
            recent[-1]['timestamp']
            - recent[0]['timestamp']
        ).seconds

        if purchase_count == 5 and time_diff < 8:

            score += 30
            reason.append("rapid_purchase")

        elif purchase_count >= 4 and time_diff < 8:

            score += 18
            reason.append(
                "rapid_purchase_pattern"
            )

    # ---------------------------------------------------
    # RULE 2: VERY FAST ACTIVITY
    # ---------------------------------------------------

    if len(user_events[user_id]) >= 2:

        last = user_events[user_id][-1]
        prev = user_events[user_id][-2]

        gap = (
            last['timestamp']
            - prev['timestamp']
        ).seconds

        if gap < 1:

            if (
                last['event_type'] == 'purchase'
                or prev['event_type'] == 'purchase'
            ):

                score += 8
                reason.append(
                    "very_fast_checkout"
                )

    # ---------------------------------------------------
    # RULE 3: PURCHASE WITHOUT BROWSING
    # ---------------------------------------------------

    if event_type == "purchase":

        last_events = user_events[user_id][-5:]

        browse_count = sum(
            1 for e in last_events
            if e['event_type'] in [
                'view',
                'cart'
            ]
        )

        purchase_count = sum(
            1 for e in last_events
            if e['event_type'] == 'purchase'
        )

        if (
            browse_count == 0
            and purchase_count >= 3
        ):

            score += 18
            reason.append(
                "purchase_without_browsing"
            )

    # ---------------------------------------------------
    # RULE 4: REPEATED PURCHASES
    # ---------------------------------------------------

    if len(recent) >= 5:

        if all(
            e['event_type'] == 'purchase'
            for e in recent
        ):

            score += 8
            reason.append(
                "repeated_purchase"
            )

    # ---------------------------------------------------
    # RULE 5: PRICE SPIKE
    # ---------------------------------------------------

    if (
        len(user_events[user_id]) >= 2
        and event_type == "purchase"
    ):

        last_price = (
            user_events[user_id][-1]['price']
        )

        prev_price = (
            user_events[user_id][-2]['price']
        )

        if last_price > prev_price * 5:

            score += 20
            reason.append("price_spike")

    # ---------------------------------------------------
    # RULE 6: LOCATION JUMP
    # ---------------------------------------------------

    if len(user_events[user_id]) >= 2:

        last_loc = (
            user_events[user_id][-1]['location']
        )

        prev_loc = (
            user_events[user_id][-2]['location']
        )

        if last_loc != prev_loc:

            location_gap = (
                user_events[user_id][-1]['timestamp']
                - user_events[user_id][-2]['timestamp']
            ).seconds

            if location_gap < 10:

                if event_type == "purchase":

                    score += 15
                    reason.append(
                        "purchase_location_jump"
                    )

                else:

                    score += 3
                    reason.append(
                        "location_change"
                    )

    # ---------------------------------------------------
    # RULE 7: MULTIPLE LOCATIONS
    # ---------------------------------------------------

    recent_locs = set(
        e['location']
        for e in user_events[user_id][-5:]
    )

    if len(recent_locs) >= 3:

        score += 5
        reason.append(
            "multiple_locations"
        )

    # ---------------------------------------------------
    # RULE 8: LOGGED OUT PURCHASE
    # ---------------------------------------------------

    if (
        event_type == "purchase"
        and not is_logged_in
    ):

        score += 15
        reason.append(
            "logged_out_purchase"
        )

    # ---------------------------------------------------
    # SCORE DECAY
    # ---------------------------------------------------

    user_scores[user_id] *= 0.85
    user_scores[user_id] += score

    total_score = round(
        user_scores[user_id],
        2
    )

    # ---------------------------------------------------
    # RISK LEVEL
    # ---------------------------------------------------

    risk_level = None

    if total_score >= 90:
        risk_level = "HIGH"

    elif total_score >= 70:
        risk_level = "MEDIUM"

    elif total_score >= 50:
        risk_level = "LOW"

    # ---------------------------------------------------
    # SAVE ALERT
    # ---------------------------------------------------

    if risk_level and len(reason) > 0:

        print(
            f"{risk_level} USER {user_id}"
            f" | Score: {total_score}"
        )

        # SAVE TO CSV
        with open(
            file_path,
            mode='a',
            newline=''
        ) as file:

            writer = csv.writer(file)

            writer.writerow([
                user_id,
                timestamp,
                total_score,
                risk_level,
                ",".join(reason)
            ])

        # SAVE TO MYSQL
        cursor.execute(
            insert_query,
            (
                user_id,
                timestamp,
                total_score,
                risk_level,
                ",".join(reason)
            )
        )

        db.commit()

        print(
            "Saved to CSV and MySQL"
        )

MySQL Connected...
Consumer started...

MEDIUM USER 13195 | Score: 71.89
Saved to CSV and MySQL
LOW USER 14052 | Score: 56.59
Saved to CSV and MySQL
LOW USER 10970 | Score: 56.59
Saved to CSV and MySQL
MEDIUM USER 9114 | Score: 73.59
Saved to CSV and MySQL
MEDIUM USER 9004 | Score: 73.59
Saved to CSV and MySQL
LOW USER 11137 | Score: 58.59
Saved to CSV and MySQL
LOW USER 13658 | Score: 56.59
Saved to CSV and MySQL
LOW USER 13126 | Score: 55.05
Saved to CSV and MySQL
LOW USER 12440 | Score: 56.59
Saved to CSV and MySQL
LOW USER 9317 | Score: 56.59
Saved to CSV and MySQL
LOW USER 14541 | Score: 56.59
Saved to CSV and MySQL
MEDIUM USER 9746 | Score: 76.59
Saved to CSV and MySQL
MEDIUM USER 10847 | Score: 76.59
Saved to CSV and MySQL
LOW USER 9026 | Score: 63.84
Saved to CSV and MySQL
LOW USER 9681 | Score: 56.59
Saved to CSV and MySQL
LOW USER 14794 | Score: 56.59
Saved to CSV and MySQL
LOW USER 13511 | Score: 56.59
Saved to CSV and MySQL
LOW USER 11071 | Score: 56.59
Saved to CSV and MyS